<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model10_Polynomial_Domain_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Drive and import required libraries
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
# Define paths for Model08/09 artifacts
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
MODEL08_DATA_PATH = RESULTS_PATH + "model08_consolidated_features.parquet"
MODEL09_FEATURE_LIST_PATH = RESULTS_PATH + "model09_recommended_feature_list.csv"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:
# Keep the FULL Model08 parquet available - needed as the source for deriving
# new nonlinear features, since some source columns (e.g. AGE_YEARS) were
# removed from the Model09 feature set during redundancy refinement.
source_df = pd.read_parquet(MODEL08_DATA_PATH)
print("Full consolidated shape:", source_df.shape)

Full consolidated shape: (307511, 402)


In [4]:
# Load Model09's 303 recommended features
model09_feature_list = pd.read_csv(MODEL09_FEATURE_LIST_PATH)
model09_features = model09_feature_list["Feature"].tolist()

print("Model09 refined feature count:", len(model09_features))

Model09 refined feature count: 303


In [5]:
# Confirm every Model09 feature is present before building the matrix
missing_model09_features = [col for col in model09_features if col not in source_df.columns]

print("Missing Model09 features:", len(missing_model09_features))
if missing_model09_features:
    for col in missing_model09_features:
        print("-", col)
    raise ValueError("Some Model09 features are missing from Model08 parquet.")

print("All Model09 features are available.")

Missing Model09 features: 0
All Model09 features are available.


In [6]:
# These source columns are needed to derive the 6 new features, even though
# AGE_YEARS was removed from the Model09 feature list itself - we are deriving
# a NEW transformation of it, not reintroducing the raw column.
required_source_features = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AGE_YEARS", "EXT_SOURCE_MEAN"]

missing_sources = [col for col in required_source_features if col not in source_df.columns]

print("Missing source columns:", len(missing_sources))
if missing_sources:
    for col in missing_sources:
        print("-", col)
    raise ValueError("Required source columns for Model10 are missing.")

print("All Model10 source columns are available.")

Missing source columns: 0
All Model10 source columns are available.


In [7]:
# X_model09: exactly the 303 Model09 features, no new columns yet
X_model09 = source_df[model09_features].copy()
y = source_df["TARGET"].copy()

print("Model09 X shape:", X_model09.shape)
print("y shape:", y.shape)

Model09 X shape: (307511, 303)
y shape: (307511,)


In [8]:
# Locked Model09 benchmark to compare against
MODEL09_ROC_AUC = 0.7872
MODEL09_PR_AUC = 0.2887

print("Locked Model09 ROC-AUC:", MODEL09_ROC_AUC)
print("Locked Model09 PR-AUC:", MODEL09_PR_AUC)

Locked Model09 ROC-AUC: 0.7872
Locked Model09 PR-AUC: 0.2887


In [9]:
# Same split used throughout the project
X_train_09, X_valid_09, y_train, y_valid = train_test_split(
    X_model09, y, test_size=0.20, random_state=42, stratify=y
)

print("Model09 training shape:", X_train_09.shape)
print("Model09 validation shape:", X_valid_09.shape)

Model09 training shape: (246008, 303)
Model09 validation shape: (61503, 303)


In [10]:
# Reusable function to build a fresh preprocessor for any feature subset
def build_preprocessor(X_data):
    numeric_cols = X_data.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_data.select_dtypes(include=["object"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

In [11]:
# Reusable function to build the same XGBoost config used throughout the project
def build_xgb():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

In [12]:
# Reusable function to train and evaluate one feature-set experiment
def run_xgb_experiment(X_train_exp, X_valid_exp, y_train_exp, y_valid_exp, experiment_name):
    print("\n" + "=" * 70)
    print(experiment_name)
    print("=" * 70)
    print("Training features:", X_train_exp.shape[1])

    preprocessor = build_preprocessor(X_train_exp)
    model = build_xgb()

    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train_exp, y_train_exp)

    valid_proba = pipeline.predict_proba(X_valid_exp)[:, 1]

    roc = roc_auc_score(y_valid_exp, valid_proba)
    pr = average_precision_score(y_valid_exp, valid_proba)

    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR-AUC:  {pr:.4f}")

    return pipeline, roc, pr

In [13]:
# Retrain the Model09 baseline here, so we have a direct in-notebook comparison
baseline_pipeline, baseline_roc_auc, baseline_pr_auc = run_xgb_experiment(
    X_train_09, X_valid_09, y_train, y_valid,
    "MODEL10 EXPERIMENT 1 - MODEL09 BASELINE"
)

print("\nModel09 locked benchmark:")
print(f"ROC-AUC: {MODEL09_ROC_AUC:.4f}")
print(f"PR-AUC:  {MODEL09_PR_AUC:.4f}")

print("\nModel09 reproduction in Model10 notebook:")
print(f"ROC-AUC: {baseline_roc_auc:.4f}")
print(f"PR-AUC:  {baseline_pr_auc:.4f}")

print("\nDifference:")
print(f"ROC-AUC: {baseline_roc_auc - MODEL09_ROC_AUC:+.6f}")
print(f"PR-AUC:  {baseline_pr_auc - MODEL09_PR_AUC:+.6f}")


MODEL10 EXPERIMENT 1 - MODEL09 BASELINE
Training features: 303
ROC-AUC: 0.7872
PR-AUC:  0.2887

Model09 locked benchmark:
ROC-AUC: 0.7872
PR-AUC:  0.2887

Model09 reproduction in Model10 notebook:
ROC-AUC: 0.7872
PR-AUC:  0.2887

Difference:
ROC-AUC: +0.000016
PR-AUC:  +0.000021


In [14]:
# Build Model10's feature matrix: start from the 303 Model09 features,
# then derive the 6 new nonlinear terms from source_df (which still has
# AGE_YEARS, EXT_SOURCE_1/2/3, and EXT_SOURCE_MEAN, even though AGE_YEARS
# itself was removed from the Model09 feature list).
model10_features = X_model09.copy()

# EXT_SOURCE squared terms: test for nonlinear relationships
model10_features["EXT_SOURCE_1_SQ"] = source_df["EXT_SOURCE_1"] ** 2
model10_features["EXT_SOURCE_2_SQ"] = source_df["EXT_SOURCE_2"] ** 2
model10_features["EXT_SOURCE_3_SQ"] = source_df["EXT_SOURCE_3"] ** 2

# Triple interaction: combined effect of all three external risk scores
model10_features["EXT_SOURCE_1_2_3"] = (
    source_df["EXT_SOURCE_1"] * source_df["EXT_SOURCE_2"] * source_df["EXT_SOURCE_3"]
)

# Age squared: tests whether default risk changes nonlinearly with age
model10_features["AGE_YEARS_SQ"] = source_df["AGE_YEARS"] ** 2

# Age x EXT_SOURCE_MEAN: tests whether external-score signal varies by age
model10_features["AGE_EXT_SOURCE_MEAN"] = source_df["AGE_YEARS"] * source_df["EXT_SOURCE_MEAN"]

new_domain_features = [
    "EXT_SOURCE_1_SQ", "EXT_SOURCE_2_SQ", "EXT_SOURCE_3_SQ",
    "EXT_SOURCE_1_2_3", "AGE_YEARS_SQ", "AGE_EXT_SOURCE_MEAN"
]

print("New Model10 features:")
for col in new_domain_features:
    print("-", col)

print("\nModel09 feature count:", X_model09.shape[1])
print("Model10 feature count:", model10_features.shape[1])

New Model10 features:
- EXT_SOURCE_1_SQ
- EXT_SOURCE_2_SQ
- EXT_SOURCE_3_SQ
- EXT_SOURCE_1_2_3
- AGE_YEARS_SQ
- AGE_EXT_SOURCE_MEAN

Model09 feature count: 303
Model10 feature count: 309


In [15]:
# Sanity-check the new features before training
display(model10_features[new_domain_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
EXT_SOURCE_1_SQ,134133.0,0.296681,0.213735,2.122305e-04,0.111561,0.256034,0.455696,0.926777
EXT_SOURCE_2_SQ,306851.0,0.301104,0.171913,6.680801e-15,0.154023,0.320312,0.440388,0.731024
EXT_SOURCE_3_SQ,246546.0,0.298935,0.188065,2.780086e-07,0.137381,0.286521,0.447637,0.802833
EXT_SOURCE_1_2_3,109589.0,0.143315,0.107598,2.430332e-07,0.056106,0.119932,0.210126,0.618557
AGE_YEARS_SQ,307511.0,2070.568868,1074.633512,4.204044e+02,1154.976271,1859.433568,2903.741862,4771.112140
AGE_EXT_SOURCE_MEAN,307339.0,22.861158,10.142511,2.017286e-04,15.086170,22.017413,30.015210,57.365341


In [16]:
# Confirm no infinities were introduced by squaring or multiplying
inf_count_model10 = np.isinf(model10_features.select_dtypes(include=np.number)).sum().sum()
print("Infinite values in Model10 features:", inf_count_model10)

Infinite values in Model10 features: 0


In [17]:
# Quick sanity check on linear relationship with TARGET
new_feature_corr = (
    pd.concat([model10_features[new_domain_features], y.rename("TARGET")], axis=1)
    .corr()["TARGET"]
    .drop("TARGET")
    .sort_values()
)

print("Correlation of new features with TARGET:")
display(new_feature_corr)

Correlation of new features with TARGET:


,TARGET
EXT_SOURCE_1_2_3,-0.188552
AGE_EXT_SOURCE_MEAN,-0.181540
EXT_SOURCE_3_SQ,-0.160346
EXT_SOURCE_2_SQ,-0.149669
EXT_SOURCE_1_SQ,-0.139878
AGE_YEARS_SQ,-0.076672


In [18]:
# Same random seed and stratification as every other split in the project
X_train_10, X_valid_10, y_train_10, y_valid_10 = train_test_split(
    model10_features, y, test_size=0.20, random_state=42, stratify=y
)

print("Model10 training shape:", X_train_10.shape)
print("Model10 validation shape:", X_valid_10.shape)

Model10 training shape: (246008, 309)
Model10 validation shape: (61503, 309)


In [19]:
# Train and evaluate with the 6 new domain features added
model10_pipeline, model10_roc_auc, model10_pr_auc = run_xgb_experiment(
    X_train_10, X_valid_10, y_train_10, y_valid_10,
    "MODEL10 EXPERIMENT 2 - MODEL09 + DOMAIN FEATURES"
)


MODEL10 EXPERIMENT 2 - MODEL09 + DOMAIN FEATURES
Training features: 309
ROC-AUC: 0.7869
PR-AUC:  0.2891


In [20]:
# Compare Model10 against both the in-notebook Model09 reproduction and the locked benchmark
roc_change = model10_roc_auc - baseline_roc_auc
pr_change = model10_pr_auc - baseline_pr_auc

roc_change_vs_model09 = model10_roc_auc - MODEL09_ROC_AUC
pr_change_vs_model09 = model10_pr_auc - MODEL09_PR_AUC

print("MODEL10 IMPROVEMENT")
print(f"ROC-AUC change vs Model09 (locked): {roc_change_vs_model09:+.4f}")
print(f"PR-AUC change vs Model09 (locked):  {pr_change_vs_model09:+.4f}")
print(f"ROC-AUC change vs Model09 (in-notebook reproduction): {roc_change:+.4f}")
print(f"PR-AUC change vs Model09 (in-notebook reproduction):  {pr_change:+.4f}")

MODEL10 IMPROVEMENT
ROC-AUC change vs Model09 (locked): -0.0003
PR-AUC change vs Model09 (locked):  +0.0004
ROC-AUC change vs Model09 (in-notebook reproduction): -0.0004
PR-AUC change vs Model09 (in-notebook reproduction):  +0.0004


In [21]:
# Build a comparison table across both experiments
model10_results = pd.DataFrame({
    "Experiment": ["Model09 Baseline", "Model10 + Domain Features"],
    "ROC-AUC": [baseline_roc_auc, model10_roc_auc],
    "PR-AUC": [baseline_pr_auc, model10_pr_auc],
    "ROC-AUC Change vs Model09": [baseline_roc_auc - MODEL09_ROC_AUC, model10_roc_auc - MODEL09_ROC_AUC],
    "PR-AUC Change vs Model09": [baseline_pr_auc - MODEL09_PR_AUC, model10_pr_auc - MODEL09_PR_AUC],
    "Feature Count": [X_train_09.shape[1], X_train_10.shape[1]]
})

display(model10_results.style.format({
    "ROC-AUC": "{:.4f}", "PR-AUC": "{:.4f}",
    "ROC-AUC Change vs Model09": "{:+.4f}", "PR-AUC Change vs Model09": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change vs Model09,PR-AUC Change vs Model09,Feature Count
0,Model09 Baseline,0.7872,0.2887,+0.0000,+0.0000,303
1,Model10 + Domain Features,0.7869,0.2891,-0.0003,+0.0004,309


In [22]:
# Keep only if PR-AUC improves AND ROC-AUC does not regress -
# protects the 0.80+ ROC-AUC target from being traded away for a small PR-AUC gain
if pr_change > 0 and roc_change >= 0:
    best_name = "Model10 + Domain Features"
    best_pipeline = model10_pipeline
    best_feature_columns = model10_features.columns.tolist()
    best_roc_auc = model10_roc_auc
    best_pr_auc = model10_pr_auc
else:
    best_name = "Model09 Baseline"
    best_pipeline = baseline_pipeline
    best_feature_columns = X_model09.columns.tolist()
    best_roc_auc = baseline_roc_auc
    best_pr_auc = baseline_pr_auc

print("Recommended experiment:", best_name)
print(f"Recommended ROC-AUC: {best_roc_auc:.4f}")
print(f"Recommended PR-AUC:  {best_pr_auc:.4f}")
print("Recommended feature count:", len(best_feature_columns))

Recommended experiment: Model09 Baseline
Recommended ROC-AUC: 0.7872
Recommended PR-AUC:  0.2887
Recommended feature count: 303


In [23]:
# Save the full experiment comparison table
model10_results.to_csv(RESULTS_PATH + "model10_polynomial_domain_results.csv", index=False)
print("Model10 results saved.")

Model10 results saved.


In [24]:
# Document each new feature and its hypothesis for the README/interview
feature_definition_table = pd.DataFrame({
    "Feature": new_domain_features,
    "Definition": [
        "EXT_SOURCE_1 ** 2", "EXT_SOURCE_2 ** 2", "EXT_SOURCE_3 ** 2",
        "EXT_SOURCE_1 * EXT_SOURCE_2 * EXT_SOURCE_3", "AGE_YEARS ** 2", "AGE_YEARS * EXT_SOURCE_MEAN"
    ],
    "Hypothesis": [
        "Nonlinear relationship of EXT_SOURCE_1 with default risk",
        "Nonlinear relationship of EXT_SOURCE_2 with default risk",
        "Nonlinear relationship of EXT_SOURCE_3 with default risk",
        "Combined nonlinear interaction of all external scores",
        "Nonlinear age-risk relationship",
        "Age may moderate the predictive effect of external scores"
    ]
})

feature_definition_table.to_csv(RESULTS_PATH + "model10_domain_feature_definitions.csv", index=False)
display(feature_definition_table)

,Feature,Definition,Hypothesis
0,EXT_SOURCE_1_SQ,EXT_SOURCE_1 ** 2,Nonlinear relationship of EXT_SOURCE_1 with de...
1,EXT_SOURCE_2_SQ,EXT_SOURCE_2 ** 2,Nonlinear relationship of EXT_SOURCE_2 with de...
2,EXT_SOURCE_3_SQ,EXT_SOURCE_3 ** 2,Nonlinear relationship of EXT_SOURCE_3 with de...
3,EXT_SOURCE_1_2_3,EXT_SOURCE_1 * EXT_SOURCE_2 * EXT_SOURCE_3,Combined nonlinear interaction of all external...
4,AGE_YEARS_SQ,AGE_YEARS ** 2,Nonlinear age-risk relationship
5,AGE_EXT_SOURCE_MEAN,AGE_YEARS * EXT_SOURCE_MEAN,Age may moderate the predictive effect of exte...


In [25]:
# Save whichever feature set won the decision above
pd.DataFrame({"Feature": best_feature_columns}).to_csv(
    RESULTS_PATH + "model10_recommended_feature_list.csv", index=False
)
print("Recommended Model10 feature list saved.")

Recommended Model10 feature list saved.


In [26]:
# Save the final benchmark for this stage
model10_benchmark = pd.DataFrame({
    "Model": ["Model10"],
    "Best_Experiment": [best_name],
    "ROC-AUC": [best_roc_auc],
    "PR-AUC": [best_pr_auc],
    "Feature_Count": [len(best_feature_columns)]
})

model10_benchmark.to_csv(RESULTS_PATH + "model10_benchmark.csv", index=False)
print("Model10 benchmark saved.")

Model10 benchmark saved.


In [27]:
# Save only the features selected for the next stage, plus target
recommended_dataset = model10_features[best_feature_columns].copy()
recommended_dataset.insert(0, "TARGET", y.values)

recommended_dataset.to_parquet(RESULTS_PATH + "model10_recommended_features.parquet", index=False)

print("Recommended Model10 dataset saved.")
print("Shape:", recommended_dataset.shape)

Recommended Model10 dataset saved.
Shape: (307511, 304)


In [28]:
# Keep only the pipeline corresponding to the chosen experiment
if best_name == "Model09 Baseline":
    del model10_pipeline
else:
    del baseline_pipeline

del source_df
gc.collect()

print("Unused experiment pipeline and source dataframe removed.")
!free -h

Unused experiment pipeline and source dataframe removed.
               total        used        free      shared  buff/cache   available
Mem:            12Gi       5.5Gi       4.0Gi       2.0Mi       3.2Gi       6.9Gi
Swap:             0B          0B          0B


In [29]:
# Install MLflow in this Colab session
!pip install mlflow -q
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [30]:
# Point at the same tracking database used throughout the project
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")
# Record this stage's parameters and metrics
with mlflow.start_run(run_name="XGBoost_Polynomial_Domain"):
    mlflow.log_param("stage", "Model10 - Polynomial Domain Features")
    mlflow.log_param("input", "Model09 redundancy-refined feature set")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("new_features", ", ".join(new_domain_features))
    mlflow.log_param("n_new_features", len(new_domain_features))
    mlflow.log_param("feature_count_before", X_model09.shape[1])
    mlflow.log_param("feature_count_after", model10_features.shape[1])
    mlflow.log_param("keep_rule", "pr_change > 0 AND roc_change >= 0")
    mlflow.log_param("selected_experiment", best_name)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("model09_baseline_roc_auc", baseline_roc_auc)
    mlflow.log_metric("model09_baseline_pr_auc", baseline_pr_auc)
    mlflow.log_metric("roc_auc", model10_roc_auc)
    mlflow.log_metric("pr_auc", model10_pr_auc)
    mlflow.log_metric("roc_auc_change_vs_Model09", roc_change_vs_model09)
    mlflow.log_metric("pr_auc_change_vs_Model09", pr_change_vs_model09)

print("Model10 logged to MLflow.")

Model10 logged to MLflow.


In [31]:
# Pull every run logged so far for comparison
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Polynomial_Domain,0.786862,0.289147
1,XGBoost_Feature_Refinement,NaN,NaN
2,XGBoost_Credit_Card,0.787019,0.287171
3,XGBoost_Installments,0.785949,0.286362
4,XGBoost_POS_CASH,0.783313,0.278581
5,XGBoost_Bureau,0.777585,0.274161
6,XGBoost_Previous_Application,0.775428,0.265853
7,XGBoost_Application_Features,0.769403,0.262725
8,XGBoost_scale_pos_weight,0.760000,0.249300
9,XGBoost_Baseline,0.761200,0.251600


In [32]:
# Print the overall before/after comparison for this stage
print("""
Model10 POLYNOMIAL / DOMAIN FEATURE EXPERIMENT COMPLETE

Model09 benchmark: ROC-AUC = 0.7872, PR-AUC = 0.2887, Features = 303
Model10 result:    ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Change vs Model09: ROC-AUC = {:+.4f}, PR-AUC = {:+.4f}

New features tested: EXT_SOURCE_1_SQ, EXT_SOURCE_2_SQ, EXT_SOURCE_3_SQ,
                      EXT_SOURCE_1_2_3, AGE_YEARS_SQ, AGE_EXT_SOURCE_MEAN

Recommended experiment: {}
Recommended feature count: {}

Next: Optuna XGBoost hyperparameter tuning, SHAP explainability, final evaluation.
""".format(model10_roc_auc, model10_pr_auc, roc_change_vs_model09, pr_change_vs_model09, best_name, len(best_feature_columns)))


Model10 POLYNOMIAL / DOMAIN FEATURE EXPERIMENT COMPLETE

Model09 benchmark: ROC-AUC = 0.7872, PR-AUC = 0.2887, Features = 303
Model10 result:    ROC-AUC = 0.7869, PR-AUC = 0.2891
Change vs Model09: ROC-AUC = -0.0003, PR-AUC = +0.0004

New features tested: EXT_SOURCE_1_SQ, EXT_SOURCE_2_SQ, EXT_SOURCE_3_SQ,
                      EXT_SOURCE_1_2_3, AGE_YEARS_SQ, AGE_EXT_SOURCE_MEAN

Recommended experiment: Model09 Baseline
Recommended feature count: 303

Next: Optuna XGBoost hyperparameter tuning, SHAP explainability, final evaluation.

